In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# 1. 路径自动对齐
project_root = Path.cwd().parent if "notebooks" in os.getcwd() else Path.cwd()
data_path = project_root / "data" / "processed" / "features_with_bsm_baseline.csv"

# 2. 读取包含多 T1 真实价格的特征大表
df = pd.read_csv(data_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
print(f" 成功载入特征矩阵，总计 {len(df)} 个交易日 ({df['Date'].min().strftime('%Y-%m-%d')} 至 {df['Date'].max().strftime('%Y-%m-%d')})")

# 3. 自动识别与生成多 T1 尺度的真实价格与定价残差
T1_SCALES = [0.25, 0.5, 0.75]

for t1 in T1_SCALES:
    cme_col = f"CME_Actual_Price_{t1}"
    bsm_col = f"Chooser_BSM_T1_{t1}"
    res_col = f"Residual_T1_{t1}"
    
    # 列名自适应兼容 (防止部分列名未写下划线)
    if cme_col not in df.columns:
        possible_cols = [c for c in df.columns if str(t1) in c and ('CME' in c or 'Actual' in c)]
        if possible_cols:
            df.rename(columns={possible_cols[0]: cme_col}, inplace=True)
            
    # 计算对应 T1 尺度的 BSM 定价残差 (Actual - BSM)
    if cme_col in df.columns and bsm_col in df.columns:
        df[res_col] = df[cme_col] - df[bsm_col]
        print(f"-> 成功加载与绑定 $T_1 = {t1}$ 年：真实价格列 [{cme_col}] 与 残差列 [{res_col}]")

# 4. 定义输入特征列 (Features Matrix)
FEATURE_COLS = [
    'JPM_Close', 'VIX_Decimal', 'Risk_Free_Rate', 'Daily_Return',
    'Rolling_Vol_20d', 'Dividend_Growth_Proxy', 'Real_Dividend_Yield',
    'VIX_JPM_Corr_20d', 'IR_Momentum_10d', 'JPM_SMA20_Disparity',
    'IV_RV_Spread', 'Rate_Delta', 
    'Chooser_BSM_T1_0.25', 'Chooser_BSM_T1_0.5', 'Chooser_BSM_T1_0.75' # 包含多尺度 BSM 基准价
]

X = df[FEATURE_COLS].copy()

# 包含 3 个尺度的真实价格 Target 与 残差 Target
Y_PRICES = df[[f"CME_Actual_Price_{t1}" for t1 in T1_SCALES if f"CME_Actual_Price_{t1}" in df.columns]].copy()
Y_RESIDUALS = df[[f"Residual_T1_{t1}" for t1 in T1_SCALES if f"Residual_T1_{t1}" in df.columns]].copy()

# 5. 严格时间序列顺序切分 (70% Train, 15% Val, 15% Test)
n_samples = len(df)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train_price, y_val_price, y_test_price = Y_PRICES.iloc[:train_end], Y_PRICES.iloc[train_end:val_end], Y_PRICES.iloc[val_end:]
y_train_res, y_val_res, y_test_res = Y_RESIDUALS.iloc[:train_end], Y_RESIDUALS.iloc[train_end:val_end], Y_RESIDUALS.iloc[val_end:]

dates_train = df['Date'].iloc[:train_end]
dates_val = df['Date'].iloc[train_end:val_end]
dates_test = df['Date'].iloc[val_end:]

print("\n" + "="*25 + "  全尺度数据集时序切分报告 " + "="*25)
print(f" 训练集 (Train 70%): {len(X_train)} 天 | {dates_train.min().strftime('%Y-%m-%d')} -> {dates_train.max().strftime('%Y-%m-%d')}")
print(f" 验证集 (Val   15%): {len(X_val)} 天 | {dates_val.min().strftime('%Y-%m-%d')} -> {dates_val.max().strftime('%Y-%m-%d')}")
print(f" 测试集 (Test  15%): {len(X_test)} 天 | {dates_test.min().strftime('%Y-%m-%d')} -> {dates_test.max().strftime('%Y-%m-%d')}")
print("="*70)

# 6. 特征标准化 (注意：仅在训练集 fit，严禁未来信息泄露)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLS, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=FEATURE_COLS, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLS, index=X_test.index)

# 7. 离线保存拟合好的 Scaler
scaler_dir = project_root / "models" / "scalers"
scaler_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, scaler_dir / "multi_t1_feature_scaler.pkl")

print("\n 特征已完成无泄露标准化。")

 成功载入特征矩阵，总计 1763 个交易日 (2018-01-02 至 2024-12-31)
-> 成功加载与绑定 $T_1 = 0.25$ 年：真实价格列 [CME_Actual_Price_0.25] 与 残差列 [Residual_T1_0.25]
-> 成功加载与绑定 $T_1 = 0.5$ 年：真实价格列 [CME_Actual_Price_0.5] 与 残差列 [Residual_T1_0.5]
-> 成功加载与绑定 $T_1 = 0.75$ 年：真实价格列 [CME_Actual_Price_0.75] 与 残差列 [Residual_T1_0.75]

=========================  全尺度数据集时序切分报告 =========================
 训练集 (Train 70%): 1234 天 | 2018-01-02 -> 2022-11-22
 验证集 (Val   15%): 264 天 | 2022-11-23 -> 2023-12-11
 测试集 (Test  15%): 265 天 | 2023-12-12 -> 2024-12-31

 特征已完成无泄露标准化。


In [7]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize_scalar
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 路径自动对齐并导入本地 bsm_chooser 模块
project_root = Path.cwd().parent if "notebooks" in os.getcwd() else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.bsm_chooser import BsmChooserPricer

# 实例化你的专属 BSM 定价器
pricer = BsmChooserPricer()
print(" 成功导入并实例化 src.models.bsm_chooser.BsmChooserPricer！")

# 2. 读取包含多 T1 真实价格的特征大表
data_path = project_root / "data" / "processed" / "features_with_bsm_baseline.csv"
df = pd.read_csv(data_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

T1_SCALES = [0.25, 0.5, 0.75]
FINAL_EXPIRY_T2 = 1.0
STRIKE = 150.0

# 3. 定义计算目标：调用 BsmChooserPricer 反解使 BSM 误差最小的最优前瞻波动率 (Opt_Sigma)
def find_optimal_sigma(row, t1):
    s0 = row['JPM_Close']
    cme_col = f'CME_Actual_Price_{t1}' if f'CME_Actual_Price_{t1}' in row else 'CME_Actual_Price'
    target = row[cme_col]
    r = row['Risk_Free_Rate']
    q = row['Real_Dividend_Yield']
    
    # 核心：调用你类中的 price_chooser 方法作为优化目标函数
    obj = lambda sig: (pricer.price_chooser(s0, STRIKE, t1, FINAL_EXPIRY_T2, r, q, sig) - target) ** 2
    res = minimize_scalar(obj, bounds=(0.001, 2.0), method='bounded')
    return res.x

print(" 正在调用 BsmChooserPricer 求解全时序最优前瞻波动率 Targets...")
for t1 in T1_SCALES:
    df[f'Opt_Sigma_{t1}'] = df.apply(lambda row: find_optimal_sigma(row, t1), axis=1)

# 4. 特征选择与严格时序切分 (70% Train, 15% Val, 15% Test)
FEATURE_COLS = [
    'JPM_Close', 'VIX_Decimal', 'Risk_Free_Rate', 'Daily_Return',
    'Rolling_Vol_20d', 'Dividend_Growth_Proxy', 'Real_Dividend_Yield',
    'VIX_JPM_Corr_20d', 'IR_Momentum_10d', 'JPM_SMA20_Disparity',
    'IV_RV_Spread', 'Rate_Delta'
]

X = df[FEATURE_COLS]
n_samples = len(df)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

X_train = X.iloc[:train_end]
X_test = X.iloc[val_end:]
df_test = df.iloc[val_end:].copy()

# 5. Approach 1 机器学习训练与二次向量化定价
print("\n" + "="*30 + "  Approach 1 测试集 (Test Set) 评估报告 " + "="*30)

app1_results = {}

for t1 in T1_SCALES:
    cme_col = f'CME_Actual_Price_{t1}' if f'CME_Actual_Price_{t1}' in df.columns else 'CME_Actual_Price'
    y_train_vol = df[f'Opt_Sigma_{t1}'].iloc[:train_end]
    
    # A. 训练 ML 波动率预测模型
    model_vol = HistGradientBoostingRegressor(random_state=42, max_iter=200, min_samples_leaf=15)
    model_vol.fit(X_train, y_train_vol)
    
    # B. 预测测试集前瞻波动率 sigma_pred
    pred_sigma_test = model_vol.predict(X_test)
    
    # C. 核心：将预测出的 sigma_pred 直接向量化传入 pricer.price_chooser 进行二次定价
    app1_price_test = pricer.price_chooser(
        s0=df_test['JPM_Close'].values,
        strike=STRIKE,
        t1=t1,
        t2=FINAL_EXPIRY_T2,
        r=df_test['Risk_Free_Rate'].values,
        q=df_test['Real_Dividend_Yield'].values,
        sigma=pred_sigma_test
    )
    
    # D. 统计与比对测试集指标
    actual_test = df_test[cme_col]
    bsm_baseline_test = df_test[f'Chooser_BSM_T1_{t1}']
    
    mae_base = mean_absolute_error(actual_test, bsm_baseline_test)
    rmse_base = np.sqrt(mean_squared_error(actual_test, bsm_baseline_test))
    
    mae_app1 = mean_absolute_error(actual_test, app1_price_test)
    rmse_app1 = np.sqrt(mean_squared_error(actual_test, app1_price_test))
    
    app1_results[t1] = {'mae': mae_app1, 'rmse': rmse_app1, 'pred_price': app1_price_test}
    
    print(f" 决策尺度 T1 = {t1} 年:")
    print(f"   ├─ BSM Baseline (20d RV) : MAE = ${mae_base:.4f} | RMSE = ${rmse_base:.4f}")
    print(f"   └─ Approach 1 (ML Vol)  : MAE = ${mae_app1:.4f} | RMSE = ${rmse_app1:.4f}")

print("="*80)
print(" 基于本地 BsmChooserPricer 的 Approach 1（间接预测法）重构成功！")

 成功导入并实例化 src.models.bsm_chooser.BsmChooserPricer！
 正在调用 BsmChooserPricer 求解全时序最优前瞻波动率 Targets...

==============================  Approach 1 测试集 (Test Set) 评估报告 ==============================
 决策尺度 T1 = 0.25 年:
   ├─ BSM Baseline (20d RV) : MAE = $10.9497 | RMSE = $11.2151
   └─ Approach 1 (ML Vol)  : MAE = $6.9657 | RMSE = $7.5705
 决策尺度 T1 = 0.5 年:
   ├─ BSM Baseline (20d RV) : MAE = $7.8358 | RMSE = $8.4064
   └─ Approach 1 (ML Vol)  : MAE = $5.1167 | RMSE = $5.5457
 决策尺度 T1 = 0.75 年:
   ├─ BSM Baseline (20d RV) : MAE = $8.6192 | RMSE = $9.2552
   └─ Approach 1 (ML Vol)  : MAE = $4.4141 | RMSE = $5.4321
 基于本地 BsmChooserPricer 的 Approach 1（间接预测法）重构成功！


In [29]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 路径自适应与数据载入
project_root = Path.cwd().parent if "notebooks" in os.getcwd() else Path.cwd()
data_path = project_root / "data" / "processed" / "features_with_bsm_baseline.csv"
df = pd.read_csv(data_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

T1_SCALES = [0.25, 0.5, 0.75]

# 2. 自动生成多 T1 尺度的真实残差目标变量 (Residual Target = CME - BSM)
for t1 in T1_SCALES:
    cme_col = f'CME_Actual_Price_{t1}' if f'CME_Actual_Price_{t1}' in df.columns else 'CME_Actual_Price'
    bsm_col = f'Chooser_BSM_T1_{t1}'
    df[f'Residual_T1_{t1}'] = df[cme_col] - df[bsm_col]

# 3. 特征定义与 70% / 15% / 15% 严格时序切分
FEATURE_COLS = [
    'JPM_Close', 'VIX_Decimal', 'Risk_Free_Rate', 'Daily_Return',
    'Rolling_Vol_20d', 'Dividend_Growth_Proxy', 'Real_Dividend_Yield',
    'VIX_JPM_Corr_20d', 'IR_Momentum_10d', 'JPM_SMA20_Disparity',
    'IV_RV_Spread', 'Rate_Delta'
]

X = df[FEATURE_COLS]
n_samples = len(df)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

X_tr = X.iloc[:train_end]
X_te = X.iloc[val_end:]
df_te = df.iloc[val_end:].copy()

# 神经网络特征标准化 (只在训练集 fit，杜绝数据泄露)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

# 4. Approach 2 训练与评估
print("\n" + "="*30 + "  Approach 2 测试集 (Test Set) 评估报告 " + "="*30)

app2_results = {}

for t1 in T1_SCALES:
    cme_col = f'CME_Actual_Price_{t1}' if f'CME_Actual_Price_{t1}' in df.columns else 'CME_Actual_Price'
    bsm_col = f'Chooser_BSM_T1_{t1}'
    
    y_tr_res = df[f'Residual_T1_{t1}'].iloc[:train_end]
    actual_test = df_te[cme_col]
    bsm_base_test = df_te[bsm_col]
    
    # A. 模型 1: GBDT 梯度提升树残差网络
    gbdt_res_model = HistGradientBoostingRegressor(random_state=42, max_iter=250, min_samples_leaf=15)
    gbdt_res_model.fit(X_tr, y_tr_res)
    pred_res_gbdt = gbdt_res_model.predict(X_te)
    price_app2_gbdt = bsm_base_test + pred_res_gbdt
    
    # B. 模型 2: ANN 全连接神经网络残差网络
    mlp_res_model = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=500,
        random_state=42,
        early_stopping=True
    )
    mlp_res_model.fit(X_tr_scaled, y_tr_res)
    pred_res_mlp = mlp_res_model.predict(X_te_scaled)
    price_app2_mlp = bsm_base_test + pred_res_mlp
    
    # 指标计算
    mae_base = mean_absolute_error(actual_test, bsm_base_test)
    rmse_base = np.sqrt(mean_squared_error(actual_test, bsm_base_test))
    
    mae_gbdt = mean_absolute_error(actual_test, price_app2_gbdt)
    rmse_gbdt = np.sqrt(mean_squared_error(actual_test, price_app2_gbdt))
    
    mae_mlp = mean_absolute_error(actual_test, price_app2_mlp)
    rmse_mlp = np.sqrt(mean_squared_error(actual_test, price_app2_mlp))
    
    app2_results[t1] = {
        'base_mae': mae_base, 'gbdt_mae': mae_gbdt, 'mlp_mae': mae_mlp,
        'gbdt_price': price_app2_gbdt, 'mlp_price': price_app2_mlp
    }
    
    print(f" 决策尺度 T1 = {t1} 年:")
    print(f"   ├─ BSM Baseline (20d RV)   : MAE = ${mae_base:.4f} | RMSE = ${rmse_base:.4f}")
    print(f"   ├─ Approach 2 (GBDT Residual): MAE = ${mae_gbdt:.4f} | RMSE = ${rmse_gbdt:.4f} (精度提升: {((mae_base-mae_gbdt)/mae_base)*100:.2f}%)")
    print(f"   └─ Approach 2 (ANN Residual) : MAE = ${mae_mlp:.4f} | RMSE = ${rmse_mlp:.4f} (精度提升: {((mae_base-mae_mlp)/mae_base)*100:.2f}%)")

print("="*80)
print(" Approach 2 完成。")


==============================  Approach 2 测试集 (Test Set) 评估报告 ==============================
 决策尺度 T1 = 0.25 年:
   ├─ BSM Baseline (20d RV)   : MAE = $10.9497 | RMSE = $11.2151
   ├─ Approach 2 (GBDT Residual): MAE = $2.2441 | RMSE = $2.6627 (精度提升: 79.51%)
   └─ Approach 2 (ANN Residual) : MAE = $11.1113 | RMSE = $13.1188 (精度提升: -1.48%)
 决策尺度 T1 = 0.5 年:
   ├─ BSM Baseline (20d RV)   : MAE = $7.8358 | RMSE = $8.4064
   ├─ Approach 2 (GBDT Residual): MAE = $2.2363 | RMSE = $2.6357 (精度提升: 71.46%)
   └─ Approach 2 (ANN Residual) : MAE = $9.1564 | RMSE = $11.8642 (精度提升: -16.85%)
 决策尺度 T1 = 0.75 年:
   ├─ BSM Baseline (20d RV)   : MAE = $8.6192 | RMSE = $9.2552
   ├─ Approach 2 (GBDT Residual): MAE = $2.9448 | RMSE = $3.5843 (精度提升: 65.83%)
   └─ Approach 2 (ANN Residual) : MAE = $7.6103 | RMSE = $8.3276 (精度提升: 11.71%)
 Approach 2 完成。
